In [ ]:
set.seed(1)

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)

In [ ]:
date <- "2026-08-28"

In [ ]:
data_dir <- "/project/work/2026-05-25_create_summary/Data/"
plot_dir <- "/project/work/2026-05-25_create_summary/Plot/"

In [ ]:
tissue_age_order <- c(
  "macula_70days","peripheral_70days",
  "macula_79days","peripheral_79days",
  "macula_87days","peripheral_87days",
  "macula_91days","peripheral_91days",
  "macula_100days","peripheral_100days",
  "macula_103days","peripheral_103days",
  "macula_116days","peripheral_116days",
  "macula_137days","peripheral_137days",
  "macula_141days","peripheral_141days",
  "macula_142days","peripheral_142days",
  "macula_162days","peripheral_162days",
  "macula_165days","peripheral_165days",
  "fovea_21years","macula_21years","peripheral_21years",
  "fovea_25years","macula_25years","peripheral_25years",
  "macula_30years","peripheral_30years",
  "fovea_32years","macula_32years","peripheral_32years",
  "fovea_67years","peripheral_67years",
  "macula_70years",
  "macula_75years","peripheral_75years",
  "macula_86years","peripheral_86years"
)

# 准备颜色

In [ ]:
library(RColorBrewer)

In [ ]:
cols.palette1 <- c(brewer.pal(9, "Set1"),brewer.pal(8, "Dark2"),brewer.pal(8, "Set2"),brewer.pal(8, "Accent"))

## age colors
cols.age <- c(cols.palette1[18:29],cols.palette1[6],cols.palette1[1:5],cols.palette1[8],cols.palette1[15])
names(cols.age) <- c('70days','79days','87days','91days','100days','103days',
                     '116days','137days','141days','142days','162days','165days',
                     '21years','25years','30years','32years',
                     '67years','70years','75years','86years')

In [ ]:
cols.palette2 <- c(brewer.pal(12, "Paired"),brewer.pal(8, "Set2"),brewer.pal(8, "Dark2"),brewer.pal(9, "Set1"))

## celltype colors
cols.celltype <- c(cols.palette2[1:8],cols.palette2[13:14])
names(cols.celltype) <- c('Rod', 'Cone', 'AC', 'BC', 'HC', 'MG', 'RGC', 'RPC','Micro','Astro')

## tissue colors
cols.tissue <- cols.palette2[9:11]
names(cols.tissue) <- c('fovea', 'macula', 'peripheral')

# 读取数据

In [ ]:
develop_meta <- readRDS("/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno_meta.rds")
adult_meta <- readRDS("/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno_meta.rds")

# 绘图

In [ ]:
plot.data <- bind_rows(develop_meta, adult_meta) |>
  dplyr::select(celltype, tissue_age) |>
  dplyr::filter(!is.na(celltype), !is.na(tissue_age)) |>
  dplyr::mutate(tissue_age = factor(tissue_age, levels = tissue_age_order)) |>
  dplyr::count(tissue_age, celltype, name = "value")

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

In [ ]:
p1 <- ggplot(plot.data, aes(x = tissue_age, y = value, fill = celltype)) +
  geom_bar(stat = "identity", position = "fill", show.legend = TRUE) +
  scale_fill_manual(name = NULL, values = cols.celltype) +
  scale_x_discrete(drop = FALSE) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
    axis.title.x = element_text(margin = margin(t = 20)),
    axis.title.y = element_text(margin = margin(r = 20)),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black"),
    axis.text.y = element_text(color = "black"),
    text = element_text(size = 15),
    axis.line.y = element_line(color = "black"),
    axis.ticks.y = element_line(color = "black"),
    plot.margin = margin(t = 5, r = 5, b = 20, l = 5)
  ) +
  labs(x = NULL, y = "Cell proportion") +
  ggtitle("celltype")

p1

In [ ]:
pdf(paste0(plot_dir,date,"_cellratio_by_tissue_age",".pdf"), width = 20, height = 10)
  print(p1)
dev.off()
paste0(plot_dir,date,"_cellratio_by_tissue_age",".pdf")

# 合并 fovea/macula 为 central 后的细胞比例图

In [ ]:
## stage 定义与 tissue_stage 顺序
stage_order <- c('Early', 'Mid', 'Late', 'Young', 'Old')
tissue_order <- c('fovea', 'macula', 'peripheral')
tissue_stage_order <- as.vector(outer(tissue_order, stage_order, function(t, s) paste(t, s, sep = "_")))

In [ ]:
## 合并 fovea/macula 为 central, 定义顺序
age_order <- c('70days','79days','87days','91days','100days','103days',
               '116days','137days','141days','142days','162days','165days',
               '21years','25years','30years','32years',
               '67years','70years','75years','86years')
tissue_merged_order <- c('central', 'peripheral')

tissue_age_merged_order <- as.vector(outer(tissue_merged_order, age_order, function(t, a) paste(t, a, sep = "_")))
tissue_stage_merged_order <- as.vector(outer(tissue_merged_order, stage_order, function(t, s) paste(t, s, sep = "_")))

In [ ]:
meta.merged <- bind_rows(develop_meta, adult_meta) |>
  dplyr::mutate(donor_age = as.character(donor_age), tissue = as.character(tissue)) |>
  dplyr::mutate(tissue_merged = ifelse(tissue %in% c("fovea", "macula"), "central", "peripheral")) |>
  dplyr::mutate(stage = dplyr::case_when(
    donor_age %in% c("70days", "79days", "87days", "91days") ~ "Early",
    donor_age %in% c("100days", "103days", "116days") ~ "Mid",
    donor_age %in% c("137days", "141days", "142days", "162days", "165days") ~ "Late",
    donor_age %in% c("21years", "25years", "30years", "32years") ~ "Young",
    donor_age %in% c("67years", "70years", "75years", "86years") ~ "Old"
  ))

In [ ]:
plot.data.age.merged <- meta.merged |>
  dplyr::mutate(tissue_age_merged = paste(tissue_merged, donor_age, sep = "_")) |>
  dplyr::select(celltype, tissue_age_merged) |>
  dplyr::filter(!is.na(celltype), !is.na(tissue_age_merged)) |>
  dplyr::mutate(tissue_age_merged = factor(tissue_age_merged, levels = tissue_age_merged_order)) |>
  dplyr::count(tissue_age_merged, celltype, name = "value") |>
  dplyr::mutate(tissue_age_merged = droplevels(tissue_age_merged))

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

In [ ]:
p1 <- ggplot(plot.data.age.merged, aes(x = tissue_age_merged, y = value, fill = celltype)) +
  geom_bar(stat = "identity", position = "fill", show.legend = TRUE) +
  scale_fill_manual(name = NULL, values = cols.celltype) +
  theme_bw() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
    axis.title.x = element_text(margin = margin(t = 20)),
    axis.title.y = element_text(margin = margin(r = 20)),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black"),
    axis.text.y = element_text(color = "black"),
    text = element_text(size = 15),
    axis.line.y = element_line(color = "black"),
    axis.ticks.y = element_line(color = "black"),
    plot.margin = margin(t = 5, r = 5, b = 20, l = 5)
  ) +
  labs(x = NULL, y = "Cell proportion") +
  ggtitle("celltype")

p1

In [ ]:
pdf(paste0(plot_dir,date,"_cellratio_by_tissue_age_central",".pdf"), width = 20, height = 10)
  print(p1)
dev.off()
paste0(plot_dir,date,"_cellratio_by_tissue_age_central",".pdf")